# UI Element Extraction on H100 (Colab) — EGM / Gemma 4 / Qwen3-VL

Three drop-in replacements for the original Gemini-based UI parser, each
running locally on a single H100 with vLLM for maximum tokens/sec.

**No flash-attn required.** The scripts pin vLLM to the `xformers`
attention backend (which ships pre-built inside the vLLM wheel) so you
don't have to deal with the flash-attn compile dance on Colab.

**Before running**: in Colab, set Runtime → Change runtime type → A100/H100.
vLLM still needs a real Hopper/Ampere GPU to hit the throughput numbers below;
T4/L4 will work but much more slowly.

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Install dependencies

vLLM 0.7+ ships its own xformers wheel, so we don't need to build
flash-attn. Transformers 4.57+ is required for Qwen3-VL and Gemma 4.

**Do NOT `pip install flash-attn`.** The scripts force the xformers
attention backend via the `VLLM_ATTENTION_BACKEND` environment variable
before vLLM is imported, so any flash-attn wheel in the env is ignored
anyway.

In [ ]:
!pip install -q --upgrade \
    "vllm>=0.7.3" \
    "transformers>=4.57.0" \
    "qwen-vl-utils>=0.0.14" \
    "pydantic>=2.0" 

## 3. (Gemma 4 only) HuggingFace login

Gemma 4 is gated. Skip this cell for EGM / Qwen3-VL.

In [ ]:
from huggingface_hub import login
login()  # paste your token

## 4. Get the scripts and your screenshots

Either upload `common.py`, `run_egm.py`, `run_gemma4.py`, `run_qwen3vl.py`
via the Files panel, or `git clone` your repo. Put your screenshots in a
directory called `Dismal/` (matching the original script's layout).

In [ ]:
!ls -lh Dismal/ | head

## 5. (Optional) Confirm flash-attn is NOT being used

The scripts set `VLLM_ATTENTION_BACKEND=XFORMERS` automatically, but if
you want belt-and-braces, set it for the whole notebook environment too:

In [ ]:
import os
os.environ['VLLM_ATTENTION_BACKEND'] = 'XFORMERS'
print('Attention backend pinned to:', os.environ['VLLM_ATTENTION_BACKEND'])

## 6a. Run EGM (recommended for UI grounding)

In [ ]:
!VLLM_ATTENTION_BACKEND=XFORMERS python run_egm.py \
    --input_dir Dismal \
    --annotated_dir Dismal_annotated_egm \
    --json_dir Dismal_json_egm

## 6b. Run Gemma 4

In [ ]:
!VLLM_ATTENTION_BACKEND=XFORMERS python run_gemma4.py \
    --model google/gemma-4-E4B-it \
    --input_dir Dismal \
    --annotated_dir Dismal_annotated_gemma4 \
    --json_dir Dismal_json_gemma4

## 6c. Run Qwen3-VL

Default uses bf16. For the throughput-optimal path, switch to the FP8
checkpoint (~1.6x more tokens/sec on H100):

In [ ]:
!VLLM_ATTENTION_BACKEND=XFORMERS python run_qwen3vl.py \
    --model Qwen/Qwen3-VL-30B-A3B-Instruct-FP8 \
    --input_dir Dismal \
    --annotated_dir Dismal_annotated_qwen3vl \
    --json_dir Dismal_json_qwen3vl

## 7. Inspect a result


In [ ]:
import json, glob
from PIL import Image
from IPython.display import display

for png in sorted(glob.glob('Dismal_annotated_egm/*.png'))[:3]:
    print(png)
    display(Image.open(png))
    js = png.replace('Dismal_annotated_egm', 'Dismal_json_egm').replace('.png', '.json')
    with open(js) as f:
        data = json.load(f)
    print(f'  {len(data["elements"])} elements')